# Pandas II: Manipulação de Tabelas

---

## 🎯 Objetivos da Aula

Ao final desta aula, o aluno será capaz de:

1. Combinar DataFrames verticalmente e horizontalmente usando `pd.concat()` com os parâmetros `axis=0` e `axis=1`.
2. Cruzar duas tabelas com `pd.merge()` usando os quatro tipos de join: `inner`, `left`, `right` e `outer`.
3. Agrupar dados por categoria e sumarizar com `groupby()` combinado a `agg()` usando funções como `sum`, `mean`, `count`, `max` e `min`.
4. Reformatar tabelas com `pivot_table()`, transformando valores de linhas em colunas.
5. Reverter a operação de pivô com `melt()`, transformando colunas em linhas.

---

## 🔥 Aquecimento — Problema Gerador (10 min)

Imagine que você é analista de dados de uma rede de lojas de roupas com filiais em São Paulo, Rio de Janeiro e Belo Horizonte. Você recebe três situações ao mesmo tempo:

**Situação 1:** Cada filial te manda um arquivo separado com as vendas do mês. Como você junta tudo em uma tabela única para analisar a rede inteira?

**Situação 2:** Você tem uma tabela de pedidos com o `id_produto` e outra tabela com os detalhes dos produtos (nome, categoria, preço). Como você cruza as duas para saber o nome de cada produto vendido — sem fazer isso linha por linha?

**Situação 3:** Você quer saber o faturamento total por categoria de produto em cada estado. Como você "gira" a tabela para que os estados virem colunas e as categorias virem linhas, igual a uma tabela dinâmica do Excel?

Essas três situações aparecem toda semana na vida de um analista de dados. E as três têm solução em Pandas com uma linha de código cada — desde que você saiba qual ferramenta usar.

Hoje você vai aprender exatamente isso: `concat`, `merge`, `groupby`, `pivot_table` e `melt`. Ao final da aula, você vai olhar para qualquer tabela e saber qual operação aplicar.




---


## 📖 Empilhando DataFrames - `concat`

`pd.concat()` junta dois ou mais DataFrames de duas formas:

- **`axis=0` (padrão):** empilha verticalmente — um embaixo do outro, como juntar folhas de papel numa pilha. As **colunas precisam ser as mesmas** (ou o Pandas preenche com NaN onde faltar).
- **`axis=1`:** junta horizontalmente — lado a lado, como colar duas colunas de planilha. Os **índices precisam ser compatíveis** (ou o Pandas preenche com NaN onde faltar).

> 📋 **Analogia do formulário de chamada:** `axis=0` é como pegar a lista de chamada da turma A e colar embaixo da lista da turma B — você tem mais alunos, mas as colunas (nome, matrícula, presença) são as mesmas. `axis=1` é como pegar a lista de presença de segunda-feira e colar do lado da de terça — os mesmos alunos, mas com uma nova coluna de data.

O parâmetro `ignore_index=True` é usado quando você não quer que o índice original apareça duplicado — ele cria um índice novo e sequencial.

### Passo 1: Preparando os dados

In [2]:
import pandas as pd
import numpy as np

# --- Dataset base: vendas de uma rede de lojas ---

# Loja São Paulo — Janeiro
vendas_sp = pd.DataFrame({
    'id_pedido'  : [101, 102, 103],
    'id_produto' : ['P001', 'P002', 'P001'],
    'quantidade' : [2, 1, 3],
    'valor'      : [299.80, 89.90, 449.70],
    'loja'       : ['SP', 'SP', 'SP']
})

# Loja Rio de Janeiro — Janeiro
vendas_rj = pd.DataFrame({
    'id_pedido'  : [201, 202, 203],
    'id_produto' : ['P003', 'P001', 'P002'],
    'quantidade' : [1, 4, 2],
    'valor'      : [199.90, 599.60, 179.80],
    'loja'       : ['RJ', 'RJ', 'RJ']
})

# Tabela de produtos (catálogo separado)
produtos = pd.DataFrame({
    'id_produto' : ['P001', 'P002', 'P003', 'P004'],
    'nome'       : ['Camiseta Básica', 'Boné', 'Vestido Floral', 'Calça Jeans'],
    'categoria'  : ['Roupas', 'Acessórios', 'Roupas', 'Roupas'],
    'preco_unit' : [149.90, 89.90, 199.90, 249.90]
})

print("Vendas SP:")
print(vendas_sp)
print("\nVendas RJ:")
print(vendas_rj)
print("\nCatálogo de Produtos:")
print(produtos)

Vendas SP:
   id_pedido id_produto  quantidade  valor loja
0        101       P001           2  299.8   SP
1        102       P002           1   89.9   SP
2        103       P001           3  449.7   SP

Vendas RJ:
   id_pedido id_produto  quantidade  valor loja
0        201       P003           1  199.9   RJ
1        202       P001           4  599.6   RJ
2        203       P002           2  179.8   RJ

Catálogo de Produtos:
  id_produto             nome   categoria  preco_unit
0       P001  Camiseta Básica      Roupas       149.9
1       P002             Boné  Acessórios        89.9
2       P003   Vestido Floral      Roupas       199.9
3       P004      Calça Jeans      Roupas       249.9


### Passo 2: `pd.concat()` — Empilhando verticalmente (axis=0)

In [3]:
# Concatenar as lojas de SP e RJ
# axis=0 (padrão) concatena verticalmente (empilha as linhas)

vendas_lojas = pd.concat([vendas_sp, vendas_rj], axis=0, ignore_index=True) # Criar um indice novo e sequencial
print("\nVendas combinadas (SP + RJ):")
display(vendas_lojas)


Vendas combinadas (SP + RJ):


,id_pedido,id_produto,quantidade,valor,loja
0,101,P001,2,299.8,SP
1,102,P002,1,89.9,SP
2,103,P001,3,449.7,SP
3,201,P003,1,199.9,RJ
4,202,P001,4,599.6,RJ
5,203,P002,2,179.8,RJ


In [4]:
# O que acontece sem ignore_index=True?

vendas_lojas = pd.concat([vendas_sp, vendas_rj], axis=0, ignore_index=False) # Mantém os índices originais, o que pode resultar em índices duplicados
print("\nVendas combinadas (SP + RJ):")
display(vendas_lojas)



Vendas combinadas (SP + RJ):


,id_pedido,id_produto,quantidade,valor,loja
0,101,P001,2,299.8,SP
1,102,P002,1,89.9,SP
2,103,P001,3,449.7,SP
0,201,P003,1,199.9,RJ
1,202,P001,4,599.6,RJ
2,203,P002,2,179.8,RJ


### Passo 3: `pd.concat()` — Juntando horizontalmente (axis=1)

In [5]:
# Dois DataFrames com informações complementares do mesmo pedido
# Simulando informações de entrega separadas das de venda
info_pedido = pd.DataFrame({
    'id_pedido' : [101, 102, 103],
    'cliente'   : ['Ana Silva', 'Bruno Costa', 'Carla Matos']
})

status_entrega = pd.DataFrame({
    'prazo_dias' : [3, 5, 2],
    'entregue'   : [True, False, True]
})

display(info_pedido)
display(status_entrega)

# Usando axis = 1 para juntar lado a lado (colunas)
pedido_completo = pd.concat([info_pedido, status_entrega], axis=1)
print("\nInformações completas do pedido:")
display(pedido_completo)


,id_pedido,cliente
0,101,Ana Silva
1,102,Bruno Costa
2,103,Carla Matos


,prazo_dias,entregue
0,3,True
1,5,False
2,2,True



Informações completas do pedido:


,id_pedido,cliente,prazo_dias,entregue
0,101,Ana Silva,3,True
1,102,Bruno Costa,5,False
2,103,Carla Matos,2,True


### Exercício 1 — `concat`
**Enunciado:** Você tem os dados de aprovação de alunos em dois semestres:

```python
sem1 = pd.DataFrame({
    'aluno'      : ['Ana', 'Bruno', 'Carla'],
    'nota_media' : [7.8, 6.5, 9.2],
    'semestre'   : [1, 1, 1]
})

sem2 = pd.DataFrame({
    'aluno'      : ['Ana', 'Bruno', 'Diego'],
    'nota_media' : [8.1, 7.0, 6.8],
    'semestre'   : [2, 2, 2]
})
```

a) Use `pd.concat()` com `axis=0` para juntar os dois semestres em uma tabela única. Use `ignore_index=True`.  
b) Quantas linhas tem a tabela final? Confirme com `.shape`.

> 💡 **Dica:** Passe os DataFrames como uma lista: `pd.concat([sem1, sem2], ...)`.

In [6]:
'''Você tem os dados de aprovação de alunos em dois semestres:

```python
sem1 = pd.DataFrame({
    'aluno'      : ['Ana', 'Bruno', 'Carla'],
    'nota_media' : [7.8, 6.5, 9.2],
    'semestre'   : [1, 1, 1]
})

sem2 = pd.DataFrame({
    'aluno'      : ['Ana', 'Bruno', 'Diego'],
    'nota_media' : [8.1, 7.0, 6.8],
    'semestre'   : [2, 2, 2]
})
```

a) Use `pd.concat()` com `axis=0` para juntar os dois semestres em uma tabela única. Use `ignore_index=True`.  
b) Quantas linhas tem a tabela final? Confirme com `.shape`.

> 💡 **Dica:** Passe os DataFrames como uma lista: `pd.concat([sem1, sem2], ...)`.'''

sem1 = pd.DataFrame({
    'aluno'      : ['Ana', 'Bruno', 'Carla'],
    'nota_media' : [7.8, 6.5, 9.2],
    'semestre'   : [1, 1, 1]
})

sem2 = pd.DataFrame({
    'aluno'      : ['Ana', 'Bruno', 'Diego'],
    'nota_media' : [8.1, 7.0, 6.8],
    'semestre'   : [2, 2, 2]
})

display(sem1)
display(sem2)

#a) Use `pd.concat()` com `axis=0` para juntar os dois semestres em uma tabela única. Use `ignore_index=True`.
sem_completo = pd.concat([sem1, sem2], axis = 0, ignore_index=True)
display(sem_completo)

#b) Quantas linhas tem a tabela final? Confirme com `.shape`.
print("Quantidade de linhas da tabela final: ", sem_completo.shape[0])

,aluno,nota_media,semestre
0,Ana,7.8,1
1,Bruno,6.5,1
2,Carla,9.2,1


,aluno,nota_media,semestre
0,Ana,8.1,2
1,Bruno,7.0,2
2,Diego,6.8,2


,aluno,nota_media,semestre
0,Ana,7.8,1
1,Bruno,6.5,1
2,Carla,9.2,1
3,Ana,8.1,2
4,Bruno,7.0,2
5,Diego,6.8,2


Quantidade de linhas da tabela final:  6


---


## 📖 Cruzando tabelas como o PROCV turbinado - `merge`

`pd.merge()` cruza dois DataFrames com base em uma **coluna em comum** (a chave), trazendo as informações de um para dentro do outro. É o equivalente ao `PROCV`/`VLOOKUP` do Excel, mas muito mais poderoso porque funciona com múltiplas correspondências.

Os quatro tipos de join definem o que acontece quando um valor existe em uma tabela mas não na outra:

| Tipo (`how=`) | O que retorna | Analogia |
|---------------|---------------|---------|
| `inner` | Só os que existem nas **duas** tabelas | Interseção de conjuntos |
| `left` | Todos da tabela **esquerda** + o que bater na direita | Tabela principal preservada |
| `right` | Todos da tabela **direita** + o que bater na esquerda | Tabela secundária preservada |
| `outer` | **Tudo** de ambas, com NaN onde não bater | União completa |

> 🧩 **Analogia da festa de casamento:** Você tem a lista de convidados (tabela da esquerda) e a lista de confirmados (tabela da direita).
> - **inner:** só quem confirmou E estava na lista de convidados.
> - **left:** todos os convidados, confirmados ou não.
> - **right:** todos que confirmaram, na lista ou não.
> - **outer:** absolutamente todo mundo dos dois lados.

O parâmetro `on=` define qual coluna é a chave. Se as colunas têm nomes diferentes nas duas tabelas, use `left_on=` e `right_on=`. O parâmetro `suffixes=` resolve conflitos quando as duas tabelas têm colunas com o mesmo nome além da chave.

### Passo 4: `pd.merge()` — Cruzando tabelas (inner join padrão)

In [7]:
# Cruzando vendas com o catálogo de produtos pela coluna 'id_produto'
# inner join (padrão): só retorna linhas que existem nas DUAS tabelas

display(vendas_lojas)
display(produtos)

vendas_inner = pd.merge(vendas_lojas, produtos, on='id_produto', how='inner')
print("\nVendas cruzadas (inner join):")
display(vendas_inner)


,id_pedido,id_produto,quantidade,valor,loja
0,101,P001,2,299.8,SP
1,102,P002,1,89.9,SP
2,103,P001,3,449.7,SP
0,201,P003,1,199.9,RJ
1,202,P001,4,599.6,RJ
2,203,P002,2,179.8,RJ


,id_produto,nome,categoria,preco_unit
0,P001,Camiseta Básica,Roupas,149.9
1,P002,Boné,Acessórios,89.9
2,P003,Vestido Floral,Roupas,199.9
3,P004,Calça Jeans,Roupas,249.9



Vendas cruzadas (inner join):


,id_pedido,id_produto,quantidade,valor,loja,nome,categoria,preco_unit
0,101,P001,2,299.8,SP,Camiseta Básica,Roupas,149.9
1,102,P002,1,89.9,SP,Boné,Acessórios,89.9
2,103,P001,3,449.7,SP,Camiseta Básica,Roupas,149.9
3,201,P003,1,199.9,RJ,Vestido Floral,Roupas,199.9
4,202,P001,4,599.6,RJ,Camiseta Básica,Roupas,149.9
5,203,P002,2,179.8,RJ,Boné,Acessórios,89.9


### Passo 5: `pd.merge()` — Os quatro tipos de join

In [8]:
# Para demonstrar os joins, vamos criar um cenário com dados ausentes

clientes = pd.DataFrame({
    'id_cliente' : [1, 2, 3, 4],
    'nome'       : ['Ana', 'Bruno', 'Carla', 'Diego']
})

pedidos_clientes = pd.DataFrame({
    'id_pedido'  : [501, 502, 503],
    'id_cliente' : [1, 2, 5],     # Cliente 5 não existe em 'clientes'
    'valor'      : [150.0, 89.0, 220.0]
    # Clientes 3 e 4 não fizeram pedidos
})

print("Clientes:")
display(clientes)
print("\nPedidos:")
display(pedidos_clientes)
print()

Clientes:


,id_cliente,nome
0,1,Ana
1,2,Bruno
2,3,Carla
3,4,Diego



Pedidos:


,id_pedido,id_cliente,valor
0,501,1,150.0
1,502,2,89.0
2,503,5,220.0


In [9]:
# --- INNER: só os que existem nos dois lados ---
inner = pd.merge(clientes, pedidos_clientes, on='id_cliente', how='inner')
print("INNER JOIN (clientes com pedidos):")
display(inner)

INNER JOIN (clientes com pedidos):


,id_cliente,nome,id_pedido,valor
0,1,Ana,501,150.0
1,2,Bruno,502,89.0


In [10]:
# --- LEFT: todos os clientes, mesmo sem pedido ---
left = pd.merge(clientes, pedidos_clientes, on='id_cliente', how='left')
print("\nLEFT JOIN (todos os clientes):")
display(left)



LEFT JOIN (todos os clientes):


,id_cliente,nome,id_pedido,valor
0,1,Ana,501.0,150.0
1,2,Bruno,502.0,89.0
2,3,Carla,NaN,NaN
3,4,Diego,NaN,NaN


In [11]:
# --- RIGHT: todos os pedidos, mesmo sem cliente cadastrado ---
right = pd.merge(clientes, pedidos_clientes, on='id_cliente', how='right')
print("\nRIGHT JOIN (todos os pedidos):")
display(right)


RIGHT JOIN (todos os pedidos):


,id_cliente,nome,id_pedido,valor
0,1,Ana,501,150.0
1,2,Bruno,502,89.0
2,5,NaN,503,220.0


In [12]:
# --- OUTER: absolutamente tudo dos dois lados ---
outer = pd.merge(clientes, pedidos_clientes, on='id_cliente', how='outer')
print("\nOUTER JOIN (todos os clientes e pedidos):")
display(outer)


OUTER JOIN (todos os clientes e pedidos):


,id_cliente,nome,id_pedido,valor
0,1,Ana,501.0,150.0
1,2,Bruno,502.0,89.0
2,3,Carla,NaN,NaN
3,4,Diego,NaN,NaN
4,5,NaN,503.0,220.0


### Passo 6: `merge` com `suffixes` — Resolvendo conflito de nomes

In [13]:
# Quando as duas tabelas têm colunas com o mesmo nome (além da chave)
tabela_a = pd.DataFrame({
    'id'    : [1, 2, 3],
    'valor' : [100, 200, 300],   # 'valor' existe nas duas tabelas!
    'fonte' : ['A', 'A', 'A']
})

tabela_b = pd.DataFrame({
    'id'    : [1, 2, 3],
    'valor' : [110, 190, 310],   # 'valor' existe nas duas tabelas!
    'fonte' : ['B', 'B', 'B']
})

# suffixes: renomeia automaticamente as colunas duplicadas
display(tabela_a)
display(tabela_b)

resultado = pd.merge(tabela_a, tabela_b, on='id', how='inner', suffixes=('_tba', '_tbb'))
display(resultado)

,id,valor,fonte
0,1,100,A
1,2,200,A
2,3,300,A


,id,valor,fonte
0,1,110,B
1,2,190,B
2,3,310,B


,id,valor_tba,fonte_tba,valor_tbb,fonte_tbb
0,1,100,A,110,B
1,2,200,A,190,B
2,3,300,A,310,B


### Exercício 2 — merge com chave em comum

**Enunciado:** Dado um sistema de RH com duas tabelas:

```python
funcionarios = pd.DataFrame({
    'id_func'  : [1, 2, 3, 4],
    'nome'     : ['Ana', 'Bruno', 'Carla', 'Diego'],
    'id_depto' : [10, 20, 10, 30]
})

departamentos = pd.DataFrame({
    'id_depto'  : [10, 20, 40],    # Depto 30 não existe; Depto 40 não tem funcionário
    'nome_depto': ['TI', 'RH', 'Marketing']
})
```

a) Faça um **inner join** para trazer o nome do departamento para a tabela de funcionários.  
b) Faça um **left join** e explique por que Diego aparece com `NaN` no departamento.

> 💡 **Dica:** A chave é `id_depto`. Use `how='inner'` e `how='left'` separadamente.


In [14]:
funcionarios = pd.DataFrame({
    'id_func'  : [1, 2, 3, 4],
    'nome'     : ['Ana', 'Bruno', 'Carla', 'Diego'],
    'id_depto' : [10, 20, 10, 30]
})

departamentos = pd.DataFrame({
    'id_depto'  : [10, 20, 40],    # Depto 30 não existe; Depto 40 não tem funcionário
    'nome_depto': ['TI', 'RH', 'Marketing']
})

#a) Faça um **inner join** para trazer o nome do departamento para a tabela de funcionários.
tab_inner = pd.merge(funcionarios, departamentos, on='id_depto', how = 'inner')
display(tab_inner)

#b) Faça um **left join** e explique por que Diego aparece com `NaN` no departamento.
tab_left = pd.merge(funcionarios, departamentos, on = 'id_depto', how = 'left')
display(tab_left)
print("Aparece NaN para o Diego, pois ele está no departamento de ID 30, mas esse departamento não está na tabela departamentos")

,id_func,nome,id_depto,nome_depto
0,1,Ana,10,TI
1,2,Bruno,20,RH
2,3,Carla,10,TI


,id_func,nome,id_depto,nome_depto
0,1,Ana,10,TI
1,2,Bruno,20,RH
2,3,Carla,10,TI
3,4,Diego,30,NaN


Aparece NaN para o Diego, pois ele está no departamento de ID 30, mas esse departamento não está na tabela departamentos


---

## 📖 Agrupar e sumarizar - `groupby` + `agg`

`groupby()` divide o DataFrame em grupos com base nos valores de uma coluna e, em seguida, aplica uma função a cada grupo. O resultado é uma tabela resumida.

> 🏆 **Analogia da tabela de classificação do Brasileirão:** Cada jogo é uma linha. `groupby('time')` agrupa todos os jogos de cada time. `agg({'pontos': 'sum', 'gols': 'sum', 'jogos': 'count'})` calcula, para cada time, o total de pontos, gols e quantidade de jogos — exatamente como a tabela de classificação do campeonato.

O fluxo padrão é:
```python
df.groupby('coluna_de_agrupamento')['coluna_de_valor'].função()
# ou, para múltiplas funções:
df.groupby('coluna_de_agrupamento').agg({'col1': 'sum', 'col2': 'mean'})
```

### Passo 7: `groupby` + `agg` — Agrupando e sumarizando

In [15]:
# Usando o DataFrame de vendas detalhadas (criado no Passo 4)
display(vendas_inner)
# Pergunta: qual o faturamento total e número de pedidos por categoria?

# --- groupby simples com uma função ---
faturamento_categoria = vendas_inner.groupby('categoria')['valor'].sum()
display(faturamento_categoria)

# --- groupby com agg: múltiplas funções ao mesmo tempo ---



# --- groupby com múltiplas colunas de agrupamento ---




,id_pedido,id_produto,quantidade,valor,loja,nome,categoria,preco_unit
0,101,P001,2,299.8,SP,Camiseta Básica,Roupas,149.9
1,102,P002,1,89.9,SP,Boné,Acessórios,89.9
2,103,P001,3,449.7,SP,Camiseta Básica,Roupas,149.9
3,201,P003,1,199.9,RJ,Vestido Floral,Roupas,199.9
4,202,P001,4,599.6,RJ,Camiseta Básica,Roupas,149.9
5,203,P002,2,179.8,RJ,Boné,Acessórios,89.9


categoria
Acessórios     269.7
Roupas        1549.0
Name: valor, dtype: float64

### Exercício 3 — groupby + agg

**Enunciado:** Uma empresa de delivery registrou os pedidos do mês:

```python
pedidos = pd.DataFrame({
    'id'        : range(1, 11),
    'cidade'    : ['SP','RJ','SP','BH','RJ','SP','BH','RJ','SP','BH'],
    'categoria' : ['Pizza','Sushi','Pizza','Hambúrguer','Sushi',
                   'Hambúrguer','Pizza','Hambúrguer','Sushi','Sushi'],
    'valor'     : [45.0, 89.0, 52.0, 38.0, 95.0, 42.0, 61.0, 35.0, 78.0, 55.0],
    'avaliacao' : [4.5, 5.0, 4.0, 3.8, 4.9, 4.2, 4.7, 3.5, 4.8, 4.6]
})
```

Responda **sem usar laços `for`**:

1. Qual é o valor total de pedidos por cidade?
2. Qual é a média de avaliação por categoria de comida?
3. Crie um resumo por cidade com: total de pedidos (`count`), valor total (`sum`) e avaliação média (`mean`). Use `.agg()` com um dicionário.

> 💡 **Dica:** Para o item 3, use `groupby('cidade').agg({'id': 'count', 'valor': 'sum', 'avaliacao': 'mean'})`.



In [16]:
pedidos = pd.DataFrame({
    'id'        : range(1, 11),
    'cidade'    : ['SP','RJ','SP','BH','RJ','SP','BH','RJ','SP','BH'],
    'categoria' : ['Pizza','Sushi','Pizza','Hambúrguer','Sushi',
                   'Hambúrguer','Pizza','Hambúrguer','Sushi','Sushi'],
    'valor'     : [45.0, 89.0, 52.0, 38.0, 95.0, 42.0, 61.0, 35.0, 78.0, 55.0],
    'avaliacao' : [4.5, 5.0, 4.0, 3.8, 4.9, 4.2, 4.7, 3.5, 4.8, 4.6]
})
display(pedidos)
# 1. Qual é o valor total de pedidos por cidade?
total_pedidos_cidade = pedidos.groupby('cidade')['valor'].sum()
display(total_pedidos_cidade)

# 2. Qual é a média de avaliação por categoria de comida?
media_avaliacao_categoria = pedidos.groupby('categoria')['avaliacao'].mean().round(2)
display(media_avaliacao_categoria)

# 3. Crie um resumo por cidade com: total de pedidos (`count`), valor total (`sum`) e avaliação média (`mean`). Use `.agg()` com um dicionário.
resumo = pedidos.groupby('cidade').agg(
    total_de_pedidos=('id', 'count'),
    valor_total=('valor', 'sum'),
    avaliacao_media=('avaliacao', 'mean')
).round(2).reset_index()
display('Tabela Resumo', resumo)

,id,cidade,categoria,valor,avaliacao
0,1,SP,Pizza,45.0,4.5
1,2,RJ,Sushi,89.0,5.0
2,3,SP,Pizza,52.0,4.0
3,4,BH,Hambúrguer,38.0,3.8
4,5,RJ,Sushi,95.0,4.9
5,6,SP,Hambúrguer,42.0,4.2
6,7,BH,Pizza,61.0,4.7
7,8,RJ,Hambúrguer,35.0,3.5
8,9,SP,Sushi,78.0,4.8
9,10,BH,Sushi,55.0,4.6


cidade
BH    154.0
RJ    219.0
SP    217.0
Name: valor, dtype: float64

categoria
Hambúrguer    3.83
Pizza         4.40
Sushi         4.82
Name: avaliacao, dtype: float64

'Tabela Resumo'

,cidade,total_de_pedidos,valor_total,avaliacao_media
0,BH,3,154.0,4.37
1,RJ,3,219.0,4.47
2,SP,4,217.0,4.38


---

## 📖 A tabela dinâmica do Pandas - `pivot_table`

`pivot_table()` reorganiza o DataFrame transformando **valores únicos de uma coluna em novas colunas**. Se você já usou tabela dinâmica no Excel, é exatamente isso — mas controlado por código.

Os parâmetros principais:
- `values`: qual coluna calcular (ex: `'vendas'`)
- `index`: qual coluna vira as linhas (ex: `'categoria'`)
- `columns`: qual coluna vira as colunas (ex: `'estado'`)
- `aggfunc`: qual função agregar (ex: `'sum'`, `'mean'`)
- `fill_value`: o que colocar quando não há valor (ex: `0`)

> 📊 **Analogia da tabela de frequências do supermercado:** Você tem uma lista gigante de compras (produto, dia da semana, valor). A pivot_table transforma isso num resumo onde as linhas são os produtos e as colunas são os dias da semana — cada célula mostra o total vendido daquele produto naquele dia. De lista para mapa, em uma linha de código.

### Passo 8: `pivot_table` — Transformando linhas em colunas

In [17]:
# Criando um dataset um pouco maior para pivot_table fazer sentido
dados_vendas = pd.DataFrame({
    'mes'       : ['Jan', 'Jan', 'Jan', 'Fev', 'Fev', 'Fev', 'Mar', 'Mar', 'Mar'],
    'categoria' : ['Roupas', 'Acessórios', 'Roupas', 'Roupas', 'Acessórios', 'Calçados',
                   'Roupas', 'Calçados', 'Acessórios'],
    'loja'      : ['SP', 'SP', 'RJ', 'SP', 'RJ', 'SP', 'RJ', 'RJ', 'SP'],
    'vendas'    : [15000, 3200, 12000, 18000, 4500, 8900, 16500, 9200, 3800]
})

print("Dados originais (formato longo):")
display(dados_vendas)
print()

# pivot_table: meses nas linhas, categorias nas colunas, soma das vendas
tabela_pivot = pd.pivot_table(
    dados_vendas,
    values = 'vendas',
    index= 'mes',
    columns= 'categoria',
    aggfunc= 'sum',
    fill_value= 0
).reset_index()
display(tabela_pivot)
# pivot_table com dois níveis de agrupamento nas linhas
tabela_pivot_2 = pd.pivot_table(
    dados_vendas,
    values = 'vendas',
    index= ['mes','loja'],
    columns= 'categoria',
    aggfunc= 'sum',
    fill_value= 0
).reset_index()
display(tabela_pivot_2)

Dados originais (formato longo):


,mes,categoria,loja,vendas
0,Jan,Roupas,SP,15000
1,Jan,Acessórios,SP,3200
2,Jan,Roupas,RJ,12000
3,Fev,Roupas,SP,18000
4,Fev,Acessórios,RJ,4500
5,Fev,Calçados,SP,8900
6,Mar,Roupas,RJ,16500
7,Mar,Calçados,RJ,9200
8,Mar,Acessórios,SP,3800


categoria,mes,Acessórios,Calçados,Roupas
0,Fev,4500,8900,18000
1,Jan,3200,0,27000
2,Mar,3800,9200,16500


categoria,mes,loja,Acessórios,Calçados,Roupas
0,Fev,RJ,4500,0,0
1,Fev,SP,0,8900,18000
2,Jan,RJ,0,0,12000
3,Jan,SP,3200,0,15000
4,Mar,RJ,0,9200,16500
5,Mar,SP,3800,0,0


### Exercício 4 — pivot_table e melt

**Enunciado:** Uma escola registrou as notas bimestrais de seus alunos:

```python
notas_raw = pd.DataFrame({
    'aluno'    : ['Ana','Ana','Ana','Ana','Bruno','Bruno','Bruno','Bruno'],
    'bimestre' : ['1B','2B','3B','4B','1B','2B','3B','4B'],
    'nota'     : [8.5, 7.0, 9.2, 8.8, 6.5, 7.5, 8.0, 7.2]
})
```

a) Crie uma `pivot_table` onde as linhas são os alunos, as colunas são os bimestres e os valores são as notas. Use `aggfunc='mean'`.  
b) Adicione uma coluna `media_anual` à tabela pivotada calculando a média de cada linha.  
c) Use `melt()` para reverter a tabela pivotada (sem a coluna `media_anual`) de volta ao formato longo.

> 💡 **Dica:** Para o item (b), use `tabela_pivot['media_anual'] = tabela_pivot.mean(axis=1)`. Para o (c), primeiro faça `reset_index()` antes do `melt()`.



In [31]:
notas_raw = pd.DataFrame({
    'aluno'    : ['Ana','Ana','Ana','Ana','Bruno','Bruno','Bruno','Bruno'],
    'bimestre' : ['1B','2B','3B','4B','1B','2B','3B','4B'],
    'nota'     : [8.5, 7.0, 9.2, 8.8, 6.5, 7.5, 8.0, 7.2]
})
display(notas_raw)
# a) Crie uma `pivot_table` onde as linhas são os alunos, as colunas são os bimestres e os valores são as notas. Use `aggfunc='mean'`.
notas_pivot = pd.pivot_table(
    notas_raw,
    values = 'nota',
    index = 'aluno',
    columns = 'bimestre',
    aggfunc = 'mean',
    fill_value = 0
)
display(notas_pivot)

# b) Adicione uma coluna `media_anual` à tabela pivotada calculando a média de cada linha.
notas_pivot['media_anual'] = notas_pivot.mean(axis = 1)
display(notas_pivot)

# c) Use `melt()` para reverter a tabela pivotada (sem a coluna `media_anual`) de volta ao formato longo.
notas_longo = notas_pivot.drop(columns='media_anual').reset_index().melt(
    id_vars='aluno', 
    var_name='bimestre', 
    value_name='nota'
)

display(notas_longo)

,aluno,bimestre,nota
0,Ana,1B,8.5
1,Ana,2B,7.0
2,Ana,3B,9.2
3,Ana,4B,8.8
4,Bruno,1B,6.5
5,Bruno,2B,7.5
6,Bruno,3B,8.0
7,Bruno,4B,7.2


bimestre,1B,2B,3B,4B
aluno,,,,
Ana,8.5,7.0,9.2,8.8
Bruno,6.5,7.5,8.0,7.2


bimestre,1B,2B,3B,4B,media_anual
aluno,,,,,
Ana,8.5,7.0,9.2,8.8,8.375
Bruno,6.5,7.5,8.0,7.2,7.300


,aluno,bimestre,nota
0,Ana,1B,8.5
1,Bruno,1B,6.5
2,Ana,2B,7.0
3,Bruno,2B,7.5
4,Ana,3B,9.2
5,Bruno,3B,8.0
6,Ana,4B,8.8
7,Bruno,4B,7.2


---
## 📖 Desfazendo o pivot (formato largo → longo) - `melt`

`melt()` faz o oposto do `pivot_table`: transforma **colunas em linhas**, convertendo uma tabela no formato "largo" (wide) para o formato "longo" (long). Muitas bibliotecas de visualização e modelos de Machine Learning exigem dados no formato longo.

Os parâmetros principais:
- `id_vars`: colunas que ficam fixas (os identificadores)
- `value_vars`: colunas que viram linhas (se omitido, usa todas as restantes)
- `var_name`: nome da nova coluna que vai receber os nomes das colunas antigas
- `value_name`: nome da nova coluna que vai receber os valores

> 🌮 **Analogia do cardápio de churrasco:** A tabela "larga" é o cardápio onde cada coluna é um item (picanha, frango, linguiça) e cada linha é uma mesa, com a quantidade pedida. `melt()` transforma isso numa lista de pedidos: mesa 1 pediu 2 picanha, mesa 1 pediu 3 frango, mesa 2 pediu 1 linguiça… — uma linha por pedido, formato longo. É o que você entrega na cozinha!

### Passo 9: `melt` — Revertendo o pivot (formato largo → longo)


In [36]:
# Vamos usar a tabela pivot criada acima e reverter para o formato longo

# Primeiro, resetamos o índice para que 'mes' vire uma coluna normal

# melt: converte colunas em linhas


## ⚠️ Erros Frequentes de Iniciantes

**Erro 1: Esquecer `ignore_index=True` no concat e ter índices duplicados**

```python
import pandas as pd

a = pd.DataFrame({'x': [1, 2]})
b = pd.DataFrame({'x': [3, 4]})

# ❌ PROBLEMA: índices 0 e 1 se repetem
errado = pd.concat([a, b])
print("Índices duplicados:", errado.index.tolist())  # [0, 1, 0, 1]

# ✅ CORRETO: regenerar o índice
certo = pd.concat([a, b], ignore_index=True)
print("Índices corretos  :", certo.index.tolist())    # [0, 1, 2, 3]
```

```
Índices duplicados: [0, 1, 0, 1]
Índices corretos  : [0, 1, 2, 3]
```

---

**Erro 2: Usar merge sem especificar `how=` e perder dados**

```python
import pandas as pd

esquerda = pd.DataFrame({'id': [1, 2, 3], 'nome': ['A', 'B', 'C']})
direita  = pd.DataFrame({'id': [1, 2],    'valor': [100, 200]})  # id=3 não existe!

# ❌ PROBLEMA: inner join padrão descarta o id=3 silenciosamente
resultado = pd.merge(esquerda, direita, on='id')
print("Inner (padrão) — id=3 sumiu:")
print(resultado)

# ✅ CORRETO: use left join para manter todos os da tabela principal
resultado_left = pd.merge(esquerda, direita, on='id', how='left')
print("\nLeft join — id=3 preservado (com NaN):")
print(resultado_left)
```

```
Inner (padrão) — id=3 sumiu:
   id nome  valor
0   1    A    100
1   2    B    200

Left join — id=3 preservado (com NaN):
   id nome  valor
0   1    A  100.0
1   2    B  200.0
2   3    C    NaN
```

---

**Erro 3: Esquecer o `reset_index()` após groupby e ter índice "estranho"**

```python
import pandas as pd

df = pd.DataFrame({
    'categoria': ['A', 'B', 'A', 'B'],
    'valor'    : [10, 20, 30, 40]
})

# ❌ CONFUSÃO: o resultado do groupby tem índice hierárquico
resultado = df.groupby('categoria')['valor'].sum()
print(type(resultado))     # Series — não é um DataFrame!
print(resultado)
print()

# ✅ CORRETO: use reset_index() para transformar em DataFrame normal
df_resultado = df.groupby('categoria')['valor'].sum().reset_index()
print(type(df_resultado))  # DataFrame
print(df_resultado)
```

```
<class 'pandas.core.series.Series'>
categoria
A    40
B    60
Name: valor, dtype: int64

<class 'pandas.core.frame.DataFrame'>
  categoria  valor
0         A     40
1         B     60
```

---

**Erro 4: Confundir pivot_table com o número de linhas esperado**

```python
import pandas as pd

df = pd.DataFrame({
    'mes'      : ['Jan', 'Jan', 'Fev'],
    'produto'  : ['A', 'A', 'A'],   # Mesmo produto em Jan aparece duas vezes!
    'vendas'   : [100, 200, 150]
})

# ❌ CONFUSÃO: sem aggfunc, o pivot falha se houver duplicatas
# pd.pivot(df, index='mes', columns='produto', values='vendas')
# → ValueError: Index contains duplicate entries

# ✅ CORRETO: usar pivot_table que agrega os duplicados automaticamente
pivot = pd.pivot_table(df, values='vendas', index='mes', columns='produto', aggfunc='sum')
print(pivot)  # Jan mostra 300 (100 + 200) — somou as duplicatas
```

```
produto    A
mes
Fev      150
Jan      300
```

---

**Erro 5: Usar melt sem especificar `id_vars` e perder as colunas identificadoras**

```python
import pandas as pd

df = pd.DataFrame({
    'produto' : ['Camiseta', 'Calça'],
    'Jan'     : [100, 200],
    'Fev'     : [150, 180]
})

# ❌ ERRADO: sem id_vars, a coluna 'produto' também vira linha!
errado = pd.melt(df)
print("Sem id_vars — 'produto' misturado com os meses:")
print(errado)
print()

# ✅ CORRETO: especificar id_vars para proteger as colunas identificadoras
certo = pd.melt(df, id_vars='produto', var_name='mes', value_name='vendas')
print("Com id_vars — 'produto' preservado:")
print(certo)
```

```
Sem id_vars — 'produto' misturado com os meses:
  variable     value
0  produto  Camiseta
1  produto     Calça
2      Jan       100
3      Jan       200
4      Fev       150
5      Fev       180

Com id_vars — 'produto' preservado:
    produto  mes  vendas
0  Camiseta  Jan     100
1     Calça  Jan     200
2  Camiseta  Fev     150
3     Calça  Fev     180
```

## Resumo Final

### 🔗 Conexão com a Próxima Aula (5 min)

Você agora sabe combinar, cruzar, agrupar e reformatar tabelas — as operações mais poderosas da análise exploratória. Mas toda análise profissional esbarra em um problema que seus dados quase certamente já têm: **valores ausentes** (campos vazios), **duplicatas** (linhas repetidas), **dados incorretos** (texto onde deveria ser número) e **categorias inconsistentes** (SP, sp, S.P. que deveriam ser a mesma coisa). Na **Aula 6**, você vai aprender a diagnosticar e corrigir todos esses problemas com as ferramentas de limpeza e transformação do Pandas — e também vai conhecer o projeto final e formar os grupos para a apresentação da Aula 7. A análise exploratória completa está quase pronta.

---

### 📚 Referências e Recursos

- 📘 [pd.concat() — documentação oficial](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- 🔗 [pd.merge() — documentação oficial com todos os tipos de join](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
- 📊 [Guia visual de joins no Pandas (com diagramas)](https://pandas.pydata.org/docs/user_guide/merging.html)
- 🏆 [groupby() — guia completo com exemplos](https://pandas.pydata.org/docs/user_guide/groupby.html)
- 🔄 [pivot_table() — documentação com exemplos](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
- 🌀 [melt() — documentação oficial](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html)
- 🎓 [Reshaping and pivot tables — guia oficial do Pandas](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- 🇧🇷 [Pandas Cheat Sheet em português (Alura)](https://www.alura.com.br/artigos/pandas-o-que-e-para-que-serve-como-instalar)
